In [122]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso

from sklearn.metrics import r2_score, mean_squared_error

In [123]:
df = pd.read_csv("cleaned_dataset1.csv")

In [124]:
df['Date_of_investing'] = pd.to_datetime(df['Date_of_investing'])

### Converting Date Column

In [125]:
df['Year'] = df['Date_of_investing'].dt.year
df['Month'] = df['Date_of_investing'].dt.month
df['Day'] = df['Date_of_investing'].dt.day

### Encode Categorical Data

In [126]:
le1 = LabelEncoder()
le2 = LabelEncoder()

df['Index_Name'] = le1.fit_transform(df['Index_Name'])
df['Country'] = le2.fit_transform(df['Country'])


In [127]:
df.drop(columns=['Date_of_investing'], inplace=True)


### Defining Features & Target

In [128]:
X = df.drop(columns=['Close'])
y = df['Close']


### Train-Test Split

In [129]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

### normalizing and scaling the data

In [130]:
x_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

In [131]:
y_scaler = StandardScaler()

y_train = y_train.values.reshape(-1,1)
y_test = y_test.values.reshape(-1,1)

y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

In [132]:
y_test_scaled

array([[-1.02941529],
       [ 0.4607271 ],
       [ 0.63196405],
       ...,
       [ 1.21790981],
       [ 1.20169289],
       [ 1.50777905]], shape=(3558, 1))

### LinearRegression

In [133]:
lr = LinearRegression()
lr.fit(X_train, y_train_scaled)

# Predict
y_pred_scaled = lr.predict(X_test)

# Convert back to original values
y_pred_lr = y_scaler.inverse_transform(y_pred_scaled)

In [134]:
y_pred_lr

array([[ 8855.70539537],
       [25389.28504613],
       [27356.2781052 ],
       ...,
       [33671.14606507],
       [33554.07797708],
       [36840.15797103]], shape=(3558, 1))

### POLYNOMIAL REGRESSION

In [135]:

poly = PolynomialFeatures(degree=2)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)


poly_scaler = StandardScaler()
X_train_poly = poly_scaler.fit_transform(X_train_poly)
X_test_poly = poly_scaler.transform(X_test_poly)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train_scaled)

y_pred_poly_scaled = poly_model.predict(X_test_poly)
y_pred_poly = y_scaler.inverse_transform(y_pred_poly_scaled)

### Ridge Regression

In [136]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train_scaled)

# Predict
y_pred_ridge_scaled = ridge.predict(X_test)

# Convert back
y_pred_ridge = y_scaler.inverse_transform(y_pred_ridge_scaled.reshape(-1,1))

### LASSO REGRESSION

In [137]:


lasso = Lasso(alpha=0.01)
lasso.fit(X_train, y_train_scaled)

# Predict
y_pred_lasso_scaled = lasso.predict(X_test)
y_pred_lasso = y_scaler.inverse_transform(y_pred_lasso_scaled.reshape(-1,1))

/home/user/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.790e+00, tolerance: 1.423e+00
  model = cd_fast.enet_coordinate_descent(


### random forest Regression

In [138]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

In [139]:


# NOTE: Random Forest doesn't need scaled y
rf.fit(X_train, y_train.ravel())

y_pred_rf = rf.predict(X_test)

## Evaluations

In [140]:

print("===== LINEAR REGRESSION =====")
print("Train R2:", r2_score(y_train, y_scaler.inverse_transform(lr.predict(X_train))))
print("Test R2:", r2_score(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))

===== LINEAR REGRESSION =====
Train R2: 0.9622927104018776
Test R2: 0.9583900488818746
MSE: 5231706.310291836


In [141]:
print("\n===== RANDOM FOREST =====")
print("Train R2:", r2_score(y_train, rf.predict(X_train)))
print("Test R2:", r2_score(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))


===== RANDOM FOREST =====
Train R2: 0.9879777603023152
Test R2: 0.9510162951808391
MSE: 6158823.8081886675


In [142]:
print("\n===== POLYNOMIAL REGRESSION =====")
print("Train R2:", r2_score(y_train, y_scaler.inverse_transform(poly_model.predict(X_train_poly))))
print("Test R2:", r2_score(y_test, y_pred_poly))
print("MSE:", mean_squared_error(y_test, y_pred_poly))


===== POLYNOMIAL REGRESSION =====
Train R2: 0.9624222317448876
Test R2: 0.9583940402047229
MSE: 5231204.472909912


In [143]:
print("\n===== RIDGE REGRESSION =====")

# Train prediction
y_train_pred_scaled = ridge.predict(X_train)
y_train_pred = y_scaler.inverse_transform(y_train_pred_scaled.reshape(-1,1))

# Test prediction already done
# y_pred_ridge

print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_pred_ridge))
print("MSE:", mean_squared_error(y_test, y_pred_ridge))


===== RIDGE REGRESSION =====
Train R2: 0.9622727027516137
Test R2: 0.9583846669399408
MSE: 5232382.993124193


In [144]:
print("\n===== LASSO REGRESSION =====")

# Train prediction
y_train_pred_scaled = lasso.predict(X_train)
y_train_pred = y_scaler.inverse_transform(y_train_pred_scaled.reshape(-1,1))

# Test prediction (already scaled → converted)
# y_pred_lasso

print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_pred_lasso))
print("MSE:", mean_squared_error(y_test, y_pred_lasso))


===== LASSO REGRESSION =====
Train R2: 0.9618473093569084
Test R2: 0.9580369381587078
MSE: 5276103.66114138
